In [ ]:
using LinearAlgebra, MPSDynamics, Statistics, FFTW, StaticArrays, DifferentialEquations
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/coupled_harmonic_oscillators_solver.jl")
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/Plot_setting.jl");

In [ ]:
function smoothstep(x)
    return x^3 * (x * (x * 6 - 15) + 10)
end

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
E_J = 90 *2pi /2  # ħ * GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
κ=1/20; # 1/ns from Jeffrey et al.

In [ ]:
function Ohmic_bath(;bath_params=(ωc_dwn=0.0*2*pi, ωc_up=10*2*pi), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params 
        ωc_dwn : lower cut-off frequency of the spectral density J(ω) in GHz
        ωc_up : upper cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    AB : Array of the segments of the frequency ranges where the spectral density J(ω) is defined.
         Each segment is an array with two elements [ω_min ω_max] in GHz.
    Jω_fct : spectral density J(ω)
    """

    ωc_dwn, ωc_up = bath_params.ωc_dwn, bath_params.ωc_up

    ### Caracterization of the impedance #####
    α = 2*κ/C_inv # no unit

    # We design the impedance to be a smooth pass-band filter
    ω_max = ωc_up-.2*2*pi # Lower cut-off of the spectral density J(ω), GHz
    ω_min = ωc_dwn+.2*2*pi # Cut-off of the spectral density J(ω), GHz


    # AB segments correspond to the different frequency ranges where the spectral density is defined.
    # It corresponds to i=1 ; i=2 and i=3 of the Jω_fct(ω,i) function.

    AB = [[-Inf ωc_dwn];[ωc_dwn ω_min];[ω_min ω_max];[ω_max ωc_up];[ωc_up Inf]]
    #AB=[[-Inf ω_min];[ω_min ω_max];[ω_max Inf]]


    function Jω_fct(ω,i)
        if i==1 # First segment of AB (here [-Inf ω_min]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [ω_min ω_max])
            y = @. smoothstep((ω - ωc_dwn) / (ω_min - ωc_dwn))
        elseif i==3 # Third segment of AB (here [ω_min ω_max])
            y = 1
        elseif i==4 # Fourth segment of AB (here [ω_max ωc_up])``
            y = @. (1 - smoothstep((ω - ω_max) / (ωc_up - ω_max)))
        elseif i==5 # Fifth segment of AB (here [ωc_up Inf])
            y = 0
        end
        return y*α.*ω
    end
    if plot_bool
        ω = range(ωc_dwn, stop=ω_max, length=1000)
        J_values = [Jω_fct(ω[i], 3)*ω[i] for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB, Jω_fct
end

function Exp_decay(;bath_params=(ω_c=50*2pi, ω_cut_off=90), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params 
        ω_c : characteristic frequency of the exponential decay in GHz
        ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    AB : Array of the segments of the frequency ranges where the spectral density J(ω) is defined.
         Each segment is an array with two elements [ω_min ω_max] in GHz.
    Jω_fct : spectral density J(ω)
    """

    ω_c, ω_cut_off = bath_params.ω_c, bath_params.ω_cut_off
    
    ### Caracterization of the impedance #####
    α = 2*κ/C_inv # no unit

    AB=[[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]

    function J_exp_decay(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @.α*ω*exp(-ω/ω_c)
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y=0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_exp_decay(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB, J_exp_decay
end

function Drude_lorentz(;bath_params=(Ω=50*2pi, ω_cut_off=100), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params
        Ω : characteristic frequency of the Drude-Lorentz function in GHz
        ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    onsite_energies : vector of on-site energies ϵ_i of the chain
    hopping_energies : vector of hopping energies t_i of the chain
    system_chain_coupling : coupling between the system and the chain
    """

    Ω, ω_cut_off = bath_params.Ω, bath_params.ω_cut_off

    AB_drude= [[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]
    function J_drude(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @. 2*κ/(C_inv)*ω*Ω^2/((ω^2+Ω^2))
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y = 0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_drude(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB_drude, J_drude
end;

In [ ]:
function One_d_non_linear(n, bath, bath_params, t_max;E_J=90 *2pi /2, κ=1/20, C_inv=0.147*2*pi, β=∞, energy=false)
    """
    INPUT :
    n : number of oscillators in the chain
    bath : function to define the spectral density J(ω) of the bath
    bath_params : parameters of the bath function
    E_J : Josephson energy in GHz (default: 90 * 2π / 2)
    κ : decay rate of the bath in GHz (default: 1/20)
    C_inv : inverse of the capacitance in GHz (default: 0.147 * 2π)
    β : inverse temperature (default: ∞ for zero temperature T=0)
    energy : boolean to indicate if the energy should be calculated (default: false)

    OUTPUT :
    ϕ_sys : array of the system's ϕ_i values at each time step
    q_sys : array of the system's q_i values at each time step
    sol.t : time vector for the simulation
    E : array of the energy values at each time step if energy=true
    """

    AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
    onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

    function damped_non_linear_chain_2ODE(ddu,du,u,p,t)
        ddu[1] = - E_J*ω_nl * sqrt(Z_nl) * sin(u[1] * sqrt(Z_nl)/ϕ_quantum) / ϕ_quantum + ω_nl * system_chain_coupling * sqrt(Z_nl) * u[2]
        temp = Mϕ_chain*u[2:end]
        temp[1] -= system_chain_coupling * sqrt(Z_nl) * u[1]
        ddu[2:end]=-Mq_chain*temp[1:end]
        #ddu[1:end] = - Mq_bis*Mϕ_bis*u[1:end]
    end

    ϕ_quantum = 1/2 # ħ/e 
    ω_nl = sqrt(4*C_inv*E_J)
    Z_nl=C_inv/ω_nl
    gq=vcat(-hopping_energies)
    gϕ=vcat(hopping_energies)
    Mq_chain=SymTridiagonal(onsite_energies, gq)
    Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)
    Mq_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([0],gq))


    #Initial conditions
    ϕ_init = 1.0
    q_init = 1.0

    ϕ_0 = vcat(ϕ_init/sqrt(Z_nl), fill(0.0, n))
    q_0 = vcat(q_init * sqrt(Z_nl), fill(0.0, n))
    ϕ_dot_0 = Mq_bis*q_0


    #Run the simulation
    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (20 * maximum(vcat(ω_nl,onsite_energies))) # Time step for the simulation
    prob = SecondOrderODEProblem(damped_non_linear_chain_2ODE, ϕ_dot_0, ϕ_0, tspan)
    sol = solve(prob, VelocityVerlet(),dt=dt)
    ϕ_dot = [getindex.(sol.u,i) for i in 1:n+1]
    ϕ = [getindex.(sol.u,i+n+1) for i in 1:n+1]

    ϕ_sys = ϕ[1] * sqrt(Z_nl)
    inv_Mq_bis = inv(Mq_bis)
    q = [inv_Mq_bis*getindex.(ϕ_dot,t) for t in 1:length(sol.t)]
    q = [getindex.(q,i) for i in 1:n+1]
    q_sys = q[1] / sqrt(Z_nl)

    if energy
        E=q_sys.^2 * C_sys_inv /2 - E_J*cos.(ϕ_sys/ϕ_quantum) - system_chain_coupling * ϕ_sys .* ϕ[2] + 1/2*[getindex.(q,i)[2:end]'*Mq_chain*getindex.(q,i)[2:end] + getindex.(ϕ,i)[2:end]'*Mϕ_chain*getindex.(ϕ,i)[2:end] for i in 1:length(sol.t)];
        return(ϕ_sys, q_sys, sol.t, E)
    end
    return(ϕ_sys, q_sys, sol.t)
end

function CL_non_linear(t_max; E_J=90 *2pi /2, κ=1/20, C_inv=0.147*2*pi)
    """
    INPUT :
    E_J : Josephson energy in GHz
    κ : damping rate in GHz
    C_inv : inverse capacitance in GHz

    OUTPUT :
    ϕ_CL : array of the time evolution of the phase ϕ(t)
    q_CL : array of the time evolution of the charge q(t)
    t : time array
    """
    # Definition of the second order ODE
    function damped_non_linear(ddu, du, u, p, t)
        ddu[1] = -2 * κ * du[1] - 4 * ϕ_quantum * E_J * C_inv *sin(u[1] / ϕ_quantum)
    end

    ϕ_quantum = 1/2 # ħ/e 
    ω_nl = sqrt(4*C_inv*E_J)
    Z_nl=C_inv/ω_nl


    # Initial conditions
    ϕ_0 = [1.]
    q_0 = [1.]
    ϕ_dot_0 = q_0 * C_inv

    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (20 * ω_nl) # Time step for the simulation

    prob = SecondOrderODEProblem(damped_non_linear,ϕ_dot_0, ϕ_0, tspan)
    sol = solve(prob, VelocityVerlet(), dt=dt)

    # Extract solution arrays
    ϕ_dot_CL = [u[1] for u in sol.u]
    ϕ_CL = [u[2] for u in sol.u]
    q_CL = ϕ_dot_CL / C_inv

    return(ϕ_CL, q_CL, sol.t)
end;

In [ ]:
function FFT_perso(ϕ,t)
    F = fftshift(fft(ϕ))
    fs = length(t)/(maximum(t)-minimum(t)) # sampling frequency
    freqs = fftshift(fftfreq(length(t), fs))
    return F, freqs
end

function comparison_simu(simu_list,E, E_params)
    ϕ_list = []
    q_list = []
    t_list = []
    E_list = []
    spectrum_list = []

    for simu in simu_list
        ϕ, q, t = simu
        push!(ϕ_list, ϕ)
        push!(q_list, q)
        push!(t_list, t)
        push!(E_list, E(ϕ, q, E_params))
        spectrum = FFT_perso(ϕ, t)
        push!(spectrum_list, spectrum)
    end

    return(E_list, spectrum_list)
end

function Energy_non_linear(ϕ, q, (E_J, C_inv))
    """
    INPUT :
    ϕ : vector of the field ϕ
    q : vector of the field q
    E_J : Josephson energy in GHz
    C_inv : inverse capacitance in GHz

    OUTPUT :
    Energy of the non-linear system
    """
    ϕ_quantum = 1/2 # ħ/e 
    return q.^2 /2 *C_inv - E_J*cos.(ϕ/ϕ_quantum)
end;

In [ ]:
function endsiteocc_d(t, cparams)
    es=cparams[1]
    ts=cparams[2]
    N=length(es)
    hmat = diagm(0=>es, 1=>ts, -1=>ts)
    F = eigen(hmat)
    U = F.vectors
    S = F.values
    real.(transpose(U[:,1].*exp.(im*t.*S))*U[:,N]*transpose(U[:,N])*(U[:,1].*exp.(-im*t.*S)))
end

function findchainlength_dich(T, cparams::Vector; eps=1e-4, verbose=false)
    Nmax = length(cparams[1])

    # Check if the full chain already fails the precision test
    occ_full = endsiteocc_d(T, [cparams[1][1:Nmax], cparams[2][1:Nmax-1]])
    occ_one_less = endsiteocc_d(T, [cparams[1][1:Nmax-1], cparams[2][1:Nmax-2]])
    if abs(occ_full - occ_one_less) > eps
        error("Suitable truncation not found, try larger starting chain length")
    end

    low = 1
    high = Nmax - 1
    best = Nmax - 1  # initialize to full chain minus 1

    while low <= high
        mid = (low + high) ÷ 2
        occ_mid = endsiteocc_d(T, [cparams[1][1:mid], cparams[2][1:mid-1]])
        occ_next = endsiteocc_d(T, [cparams[1][1:mid+1], cparams[2][1:mid]])

        verbose && println("Testing ntrunc = $mid")

        if abs(occ_mid - occ_next) > eps
            low = mid + 1  # too short: try longer
        else
            best = mid     # found acceptable length
            high = mid - 1 # try even shorter
        end
    end

    return best
end;

# Ohmic bath

In [ ]:
n_huge=700
AB, J = Ohmic_bath(bath_params=(ωc_dwn=0.0, ωc_up=30*2*pi),κ=1/20, C_inv=0.147*2*pi)
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
n

In [ ]:
ϕ_1d, q_1d, t_1d = One_d_non_linear(n, Ohmic_bath, (ωc_dwn=0.0, ωc_up=10*2*pi), t_max);

In [ ]:
ϕ_CL, q_CL, t_CL = CL_non_linear(t_max)

In [ ]:
plot(ϕ_1d, q_1d, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_ohm_non_linear.pdf")

In [ ]:
plot(t_1d, ϕ_1d, xlabel="t", ylabel="ϕ", label="")

In [ ]:
AB, J = Ohmic_bath(bath_params=(ωc_dwn=0.0, ωc_up=10*2*pi),κ=1/20, C_inv=0.147*2*pi)
params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)

In [ ]:
scatter(params[1])

In [ ]:
ωc_up = 10*2*pi
(ωc_up^2/(2C_inv))/(2pi)

In [ ]:
sqrt(4*90*2*pi*C_inv)

# Ohmic bath with exponential decay

In [ ]:
n_huge=1200
AB, J = Exp_decay()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
ϕ_1d_e, q_1d_e, t_1d_e = One_d_non_linear(n, Exp_decay, (ω_c=50*2pi, ω_cut_off=90), t_max, E_J=90*2*pi/2);

In [ ]:
plot(ϕ_1d_e, q_1d_e, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_exp_decay_non_linear.pdf")

In [ ]:
plot(t_1d_e, ϕ_1d_e, xlabel="t", ylabel="ϕ", label="")

# Drude-Lorentz 

In [ ]:
n_huge=1100
AB, J = Drude_lorentz()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
ϕ_1d_l, q_1d_l, t_1d_l = One_d_non_linear(n, Drude_lorentz, (Ω=50*2pi, ω_cut_off=100), t_max);

In [ ]:
plot(ϕ_1d_l, q_1d_l, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_drude_lorentz_non_linear.pdf")

## Comparison

In [ ]:
simu=(ϕ_1d, q_1d, t_1d)
simu_CL=(ϕ_CL, q_CL, t_CL)
simu_e=(ϕ_1d_e, q_1d_e, t_1d_e)
simu_DL=(ϕ_1d_l, q_1d_l, t_1d_l)
E_list, spectrum_list = comparison_simu([simu, simu_CL, simu_e, simu_DL], Energy_non_linear, (E_J, C_inv))
E = E_list[1]
E_CL = E_list[2]
E_e = E_list[3]
E_DL = E_list[4]
(F_1d,freqs_1d) = spectrum_list[1]
(F_CL,freqs_CL) = spectrum_list[2]
(F_e,freqs_e) = spectrum_list[3]
(F_DL,freqs_DL) = spectrum_list[4];

In [ ]:
plot(freqs_CL, abs.(F_CL), label = "CL master equation")
plot!(freqs_1d, abs.(F_1d), label = "Ohmic bath")
plot!(freqs_e, abs.(F_e), label = "Ohmic bath exp decay")
plot!(freqs_DL, abs.(F_DL), label = "Drude-Lorentz bath")
plot!(xlims=(3,8), xlabel="Frequency (GHz)", ylabel="Amplitude")
#savefig("frequency_comparison_CL_baths.pdf")

In [ ]:
theoretical_decay = exp.(-2κ*t_CL)*(E_CL[1]+E_J)

In [ ]:
plot(t_CL.*κ, E_CL.+E_J, label="CL master equation")
plot!(t_1d.*κ, E.+E_J, label="Ohmic bath")
plot!(t_1d_e.*κ, E_e.+E_J, label="Ohmic bath exp decay")
plot!(t_1d_l.*κ, E_DL.+E_J, label="Drude-Lorentz bath")
plot!(t_CL.*κ, theoretical_decay, label=L"E(0)\exp(-2\kappa t)", linestyle=:dash)
plot!(xlabel=L"\kappa t", ylabel="Energy (GHz)", yaxis=:log10) 
#savefig("energy_comparison_CL_baths.pdf")

## Driven oscillator

In [ ]:
function One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=90 *2pi /3, κ=1/20, C_inv=0.147*2*pi, ω=sqrt(4*C_inv*E_J), A=5, β=∞, energy=false)
    """
    INPUT :
    n : number of oscillators in the chain
    bath : function to define the spectral density J(ω) of the bath
    bath_params : parameters of the bath function
    E_J : Josephson energy in GHz (default: 90 * 2π / 3)
    κ : decay rate of the bath in GHz (default: 1/20)
    C_inv : inverse of the capacitance in GHz (default: 0.147 * 2π)
    ω : frequency of the driving force in GHz (resonant by default : ω_nl = sqrt(4*C_inv*E_J))
    A : amplitude of the driving force (default: 5)
    β : inverse temperature (default: ∞ for zero temperature T=0)
    energy : boolean to indicate if the energy should be calculated (default: false)

    OUTPUT :
    ϕ_sys : array of the system's ϕ_i values at each time step
    q_sys : array of the system's q_i values at each time step
    sol.t : time vector for the simulation
    E : array of the energy values at each time step if energy=true
    """

    AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
    onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

    function damped_non_linear_chain_2ODE_driven(ddu,du,u,p,t)
        ddu[1] = - E_J*ω_nl * sqrt(Z_nl) * sin(u[1] * sqrt(Z_nl)/ϕ_quantum) / ϕ_quantum + ω_nl * system_chain_coupling * sqrt(Z_nl) * u[2] + A*cos(ω * t) / sqrt(Z_nl)
        #ddu[1] = - E_J*ω_nl * sqrt(Z_nl) * u[1] * sqrt(Z_nl)/ϕ_quantum / ϕ_quantum + ω_nl * system_chain_coupling * sqrt(Z_nl) * u[2] + A*cos(ω * t) / sqrt(Z_nl) #if we want to use the linearized equation
        temp = Mϕ_chain*u[2:end]
        temp[1] -= system_chain_coupling * sqrt(Z_nl) * u[1]
        ddu[2:end]=-Mq_chain*temp[1:end]
        #ddu[1:end] = - Mq_bis*Mϕ_bis*u[1:end] # if we want to use the linearized equation in a more robust way
    end

    ϕ_quantum = 1/2 # ħ/e 
    ω_nl = sqrt(4*C_inv*E_J)
    Z_nl=C_inv/ω_nl
    gq=vcat(-hopping_energies)
    gϕ=vcat(hopping_energies)
    Mq_chain=SymTridiagonal(onsite_energies, gq)
    Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)
    Mq_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([0],gq))
    Mϕ_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([-system_chain_coupling * sqrt(Z_nl)],gϕ))

    #Initial conditions
    ϕ_init = 1.0
    q_init = 1.0

    ϕ_0 = vcat(ϕ_init/sqrt(Z_nl), fill(0.0, n))
    q_0 = vcat(q_init * sqrt(Z_nl), fill(0.0, n))
    ϕ_dot_0 = Mq_bis*q_0


    #Run the simulation
    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (40 * maximum(vcat(ω_nl,onsite_energies))) # Time step for the simulation
    prob = SecondOrderODEProblem(damped_non_linear_chain_2ODE_driven, ϕ_dot_0, ϕ_0, tspan)
    sol = solve(prob, VelocityVerlet(),dt=dt)
    ϕ_dot = [getindex.(sol.u,i) for i in 1:n+1]
    ϕ = [getindex.(sol.u,i+n+1) for i in 1:n+1]

    ϕ_sys = ϕ[1] * sqrt(Z_nl)
    inv_Mq_bis = inv(Mq_bis)
    q = [inv_Mq_bis*getindex.(ϕ_dot,t) for t in 1:length(sol.t)]
    q = [getindex.(q,i) for i in 1:n+1]
    q_sys = q[1] / sqrt(Z_nl)

    if energy
        E=q_sys.^2 * C_inv /2 - E_J*cos.(ϕ_sys/ϕ_quantum) - system_chain_coupling * ϕ_sys .* ϕ[2] + 1/2*[getindex.(q,i)[2:end]'*Mq_chain*getindex.(q,i)[2:end] + getindex.(ϕ,i)[2:end]'*Mϕ_chain*getindex.(ϕ,i)[2:end] for i in 1:length(sol.t)];
        return(ϕ_sys, q_sys, sol.t, E)
    end
    return(ϕ_sys, q_sys, sol.t)
end

function CL_non_linear_driven(t_max;E_J=90 *2pi /3, κ=1/20, C_inv=0.147*2*pi, ω=sqrt(4*C_inv*E_J), A=5)
    """
    INPUT :
    E_J : Josephson energy in GHz
    κ : damping rate in GHz
    C_inv : inverse capacitance in GHz
    ω : frequency of the driving force in GHz (resonant by default : ω_nl = sqrt(4*C_inv*E_J))
    A : amplitude of the driving force (default: 5)

    OUTPUT :
    ϕ_CL : array of the time evolution of the phase ϕ(t)
    q_CL : array of the time evolution of the charge q(t)
    t : time array
    """
    # Definition of the second order ODE
    function damped_non_linear_driven(ddu, du, u, p, t)
        ddu[1] = -2 * κ * du[1] - E_J * C_inv *sin(u[1] / ϕ_quantum) / ϕ_quantum + A * cos(ω * t)
        #ddu[1] = -2 * κ * du[1] - E_J * C_inv *u[1] / ϕ_quantum / ϕ_quantum + A * cos(ω * t)
    end

    
    ϕ_quantum = 1/2 # ħ/e 
    ω_nl = sqrt(4*C_inv*E_J)
    Z_nl=C_inv/ω_nl


    # Initial conditions
    ϕ_0 = [1.]
    q_0 = [1.]
    ϕ_dot_0 = q_0 * C_inv

    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (60 * ω_nl) # Time step for the simulation

    prob = SecondOrderODEProblem(damped_non_linear_driven,ϕ_dot_0, ϕ_0, tspan)
    sol = solve(prob, VelocityVerlet(), dt=dt)

    # Extract solution arrays
    ϕ_dot_CL = [u[1] for u in sol.u]
    ϕ_CL = [u[2] for u in sol.u]
    q_CL = ϕ_dot_CL / C_inv

    return(ϕ_CL, q_CL, sol.t)
end;

### Ohmic bath

In [ ]:
n_huge=1000
AB, J = Ohmic_bath(bath_params=(ωc_dwn=0.0, ωc_up=30*2*pi),κ=1/20, C_inv=0.147*2*pi)
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
n_huge=2000
AB, J = Ohmic_bath(bath_params=(ωc_dwn=0.0, ωc_up=30*2*pi),κ=1/20, C_inv=0.147*2*pi)
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
ϕ_1d_test, q_1d_test, t_1d_test = One_d_non_linear_driven(n, Ohmic_bath, (ωc_dwn=0.0*2pi, ωc_up=10.0*2pi), t_max, energy=false);

In [ ]:
plot(ϕ_1d_test, q_1d_test, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_ohm_non_linear_driven.pdf")

In [ ]:
ϕ_CL_test, q_CL_test, t_CL_test = CL_non_linear_driven(t_max);

In [ ]:
plot(ϕ_CL_test, q_CL_test, xlabel="ϕ", ylabel="q", label="")
#savefig("CL_non_linear_driven.pdf")

In [ ]:
A=1
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
E_J = 90 *2pi /2  # ħ * GHz
κ=1/20; # 1/ns from Jeffrey et al.
ω_nl = sqrt(4*C_inv*E_J)
ω=ω_nl
Ω=sqrt(ω_nl^2-κ^2)
t=t_CL_test
ϕ_0 = [1.]
q_0 = [1.]
ϕ_dot_0 = q_0 * C_inv
ϕ_p_ana = @. A *(ω_nl^2-ω^2)/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)*cos(ω_nl * t) +
            A * 2 * κ * ω_nl/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)*sin(ω_nl * t)
ϕ_dot_0_h = ϕ_dot_0[1] - A * 2 * κ * ω_nl/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2) * ω_nl
ϕ_0_h = ϕ_0[1] - A * (ω_nl^2-ω^2)/((ω_nl^2-ω^2)^2+(2 * κ * ω_nl)^2)
ϕ_h_ana = @. exp(-κ*t)*(ϕ_0_h*cos(Ω*t) + (ϕ_dot_0_h+κ*ϕ_0_h)*sin(Ω*t)/Ω)

ϕ_ana_tot = ϕ_p_ana + ϕ_h_ana;

In [ ]:
plot(t_1d_test.*κ, ϕ_1d_test, label="ϕ 1d")
plot!(t_CL_test.*κ, ϕ_CL_test, label="ϕ CL")
#plot!(t_CL_test.*κ, ϕ_ana_tot, label="ϕ analytical")
#plot!(xlim=(0, 0.1), xlabel=L"\kappa t")

### Exp decay

In [ ]:
n_huge=1000
AB, J = Exp_decay()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
n_huge=2000
AB, J = Exp_decay()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
ϕ_1d_test_e, q_1d_test_e, t_1d_test_e = One_d_non_linear_driven(n, Exp_decay, (ω_c=50*2pi, ω_cut_off=90), t_max);

In [ ]:
plot(ϕ_1d_test_e, q_1d_test_e, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_exp_decay_non_linear_driven.pdf")

In [ ]:
plot(t_1d_test_e.*κ, ϕ_1d_test_e, label="ϕ 1d")
plot!(t_CL_test.*κ, ϕ_CL_test, label="ϕ CL")

### Drude-Lorentz

In [ ]:
n_huge=1100
AB, J = Drude_lorentz()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
n_huge=2000
AB, J = Drude_lorentz()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
ϕ_1d_test_DL, q_1d_test_DL, t_1d_test_DL = One_d_non_linear_driven(n, Drude_lorentz, (Ω=50*2pi, ω_cut_off=100), t_max);

In [ ]:
plot(ϕ_1d_test_DL, q_1d_test_DL, xlabel="ϕ", ylabel="q", label="")
#savefig("1d_chain_DL_non_linear_driven.pdf")

In [ ]:
plot(t_1d_test_DL.*κ, ϕ_1d_test_DL, label="ϕ 1d")
plot!(t_CL_test.*κ, ϕ_CL_test, label="ϕ CL")

In [ ]:
simu=(ϕ_1d_test, q_1d_test, t_1d_test)
simu_CL=(ϕ_CL_test, q_CL_test, t_CL_test)
simu_e=(ϕ_1d_test_e, q_1d_test_e, t_1d_test_e)
simu_DL=(ϕ_1d_test_DL, q_1d_test_DL, t_1d_test_DL)
E_list, spectrum_list = comparison_simu([simu, simu_CL, simu_e, simu_DL], Energy_non_linear, (E_J, C_inv))
E = E_list[1]
E_CL = E_list[2]
E_e = E_list[3]
E_DL = E_list[4]
(F_1d_test,freqs_1d_test) = spectrum_list[1]
(F_CL_test,freqs_CL_test) = spectrum_list[2]
(F_e,freqs_e) = spectrum_list[3]
(F_DL,freqs_DL) = spectrum_list[4];

In [ ]:
plot(t_1d_test.*κ, E, label="1d-chain")
plot!(t_CL_test.*κ, E_CL, label="CL")
plot!(t_1d_test_e.*κ, E_e, label="1d-chain exp decay")
plot!(t_1d_test_DL.*κ, E_DL, label="1d-chain Drude-Lorentz")
savefig("energy_comparison_CL_baths_driven_non_linear.pdf")

In [ ]:
plot(freqs_1d_test, abs.(F_1d_test), label = "1d-chain")
plot!(freqs_CL_test, abs.(F_CL_test), label = "CL")
plot!(freqs_e, abs.(F_e), label = "1d-chain exp decay")
plot!(freqs_DL, abs.(F_DL), label = "1d-chain Drude-Lorentz")
plot!(xlims=(2,8), xlabel="Frequency (GHz)", ylabel="Amplitude")
savefig("frequency_comparison_CL_baths_driven_non_linear.pdf")

In [ ]:
E_J=90*2*pi

In [ ]:
sqrt(4*C_inv*E_J)/(2*pi)